# 레슨 4-1. 모형 평가: 변별력·보정·혼동행렬

**예상 시간** 90분 · **선행** 레슨 3-2 · **이론 페이지** `docs/theory/stage_09.html`

## 학습 목표
1. 8건짜리 예제로 **AUROC·Gini·KS** 를 손으로 계산하고 라이브러리 값과 맞춘다.
2. AUROC 가 **부도–정상 쌍의 순위 비교 확률**임을 직접 확인한다.
3. 혼동행렬 지표가 **임계값(컷오프)에 따라 달라지고**, 부도율이 낮을 때 정확도가 속임수가 됨을 본다.
4. **변별력과 보정은 다르다**: 확률에 2를 곱해도 AUROC 는 그대로지만 보정은 깨진다.
5. 지표의 **불확실성**(부트스트랩 구간)과 **기간별 변동**을 확인한다.

> **약속**: 평가는 **테스트 구간에서만** 합니다. 개발 구간의 지표는 *성능*이 아니라 *적합도*입니다.
> 합성데이터 결과는 실제 성능이 아닙니다. 이 레슨은 교육용입니다.

In [ ]:
# [환경 준비] 맨 위 셀부터 순서대로 실행하세요. (Colab/로컬 모두 동작)
import os
import subprocess
import sys
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/nh15312323-art/CreditModel.git"


def find_or_fetch_project() -> Path:
    for base in (Path.cwd(), Path.cwd().parent, Path("/content/credit-model-mastery-lab")):
        if (base / "src" / "creditlab").exists():
            return base
    if not IN_COLAB:
        raise RuntimeError("프로젝트 폴더를 찾지 못했습니다. 저장소 루트(또는 notebooks/)에서 노트북을 여세요.")
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/credit-model-mastery-lab"], check=True)
    else:
        from google.colab import files  # Colab 전용

        print("credit-model-mastery-lab.zip 파일을 선택해 업로드하세요.")
        files.upload()
        subprocess.run(["unzip", "-q", "-o", "credit-model-mastery-lab.zip", "-d", "/content"], check=True)
    return Path("/content/credit-model-mastery-lab")


PROJECT_ROOT = find_or_fetch_project()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
if IN_COLAB:  # UCI 원본이 엑셀(.xls)이라 읽기 엔진이 필요합니다
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xlrd", "openpyxl"], check=False)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import creditlab
from creditlab import environment_info, set_seed

SEED = set_seed(42)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
print("프로젝트 위치:", PROJECT_ROOT)
print("creditlab", creditlab.__version__, "| 시드", SEED)
print({k: v for k, v in environment_info(SEED)["packages"].items()})

## 1. 세 가지 질문을 따로 본다
| 질문 | 지표 |
|---|---|
| 위험한 사람을 **더 위에 세우는가**(순위) | AUROC, Gini, KS |
| 예측 **확률 값이 맞는가** | Brier, log loss, 보정표 |
| **특정 컷오프에서** 어떤 오류가 나는가 | 혼동행렬(정밀도·재현율 …) |

규칙: 이 프로그램의 평가 함수는 `risk`(클수록 **위험**)를 받습니다. PD 는 그대로, 스코어카드 점수는 **`−점수`** 를 넣습니다.

## 2. 손으로 풀어 보는 8건 예제

In [ ]:
pred_auc = None  # ← 아래 8건의 AUROC 를 예측해 보세요 (부도 3건 × 정상 5건 = 15쌍 중 부도가 위인 쌍 비율)

In [ ]:
from creditlab.evaluation import (auroc, gini, ks_statistic, cap_accuracy_ratio, brier_score, log_loss_score, confusion_at)

s = np.array([0.9, 0.7, 0.4, 0.8, 0.5, 0.3, 0.2, 0.1])      # 예측 부도확률
y = np.array([1, 1, 1, 0, 0, 0, 0, 0])
print(pd.DataFrame({"y": y, "score": s}).sort_values("score", ascending=False).T.to_string())

bad, good = s[y == 1], s[y == 0]
pairs = [(b > g) + 0.5 * (b == g) for b in bad for g in good]
print(f"\n쌍 비교: {int(sum(pairs))}/{len(pairs)} = {np.mean(pairs):.3f}   라이브러리 AUROC = {auroc(y, s):.3f}")
print(f"Gini = 2×AUROC−1 = {gini(y, s):.3f},  KS = {ks_statistic(y, s)['ks']:.3f} (임계값 {ks_statistic(y, s)['threshold']})")
print(f"Brier = {brier_score(y, s):.5f},  log loss = {log_loss_score(y, s):.4f}")
if pred_auc is not None:
    print("내 예측:", pred_auc)

**읽는 법**: 부도 0.9·0.7 은 모든 정상보다 위, 부도 0.4 는 정상 0.8·0.5 보다 아래입니다. 쌍 15개 중 12개가 맞아 AUROC 0.8 입니다. *임계값이 필요 없다*는 점이 혼동행렬 지표와 다릅니다.

## 3. 데이터·모형 (레슨 3-2 의 절차를 짧게)

In [ ]:
from creditlab.data import load_learning_data
from creditlab.features import fit_binners, transform_woe_frame
from creditlab.models import fit_logistic
from creditlab.scorecard import ScoreScaling, Scorecard

df_all, meta = load_learning_data("synthetic", seed=SEED)
TARGET = meta["target"]
df_all = df_all.sort_values("application_date").reset_index(drop=True)
cut = int(len(df_all) * 0.7)
dev, test = df_all.iloc[:cut].copy(), df_all.iloc[cut:].copy()

VARS = ["num_delinq_12m", "card_utilization", "age", "inquiries_6m", "employment_years", "loan_amount_10k"]
binners = fit_binners(dev, VARS, TARGET, method="quantile", max_bins=5, min_events=10)
fit = fit_logistic(transform_woe_frame(dev, binners), dev[TARGET])
card = Scorecard.from_fit(binners, fit, scaling=ScoreScaling(600, 50, 20))

pd_dev, pd_test = card.predict_pd(dev), card.predict_pd(test)
sc_dev, sc_test = card.score(dev), card.score(test)
print(f"개발 {len(dev):,}행 부도 {int(dev[TARGET].sum())}건 ({dev[TARGET].mean():.1%}) | 테스트 {len(test):,}행 부도 {int(test[TARGET].sum())}건 ({test[TARGET].mean():.1%})")

## 4. 변별력 — 점수와 PD 는 같은 순위

In [ ]:
from creditlab.evaluation import discrimination_summary

a_pd = auroc(test[TARGET], pd_test)
a_score = auroc(test[TARGET], -sc_test)            # 점수는 높을수록 안전 → 위험 방향으로 부호 반전
a_wrong = auroc(test[TARGET], sc_test)             # 방향을 잘못 넣으면?
print(f"AUROC (PD)        = {a_pd:.4f}")
print(f"AUROC (−점수)     = {a_score:.4f}   ← 같아야 한다 (단조 변환은 순위를 바꾸지 않음)")
print(f"AUROC (+점수, 오류)= {a_wrong:.4f}   ← 방향을 뒤집으면 1−AUROC = {1 - a_pd:.4f}")

res = pd.DataFrame({"개발": discrimination_summary(dev[TARGET], pd_dev), "테스트": discrimination_summary(test[TARGET], pd_test)}).T
print()
print(res[["n", "bads", "bad_rate", "auroc", "gini", "ks", "avg_precision"]].round(4).to_string())

- **방향을 뒤집으면 AUROC 가 `1−AUROC`** 가 됩니다. 점수를 그대로 넣는 실수는 `0.5 미만`의 숫자로 드러납니다.
- 개발 구간 지표는 그 데이터에 맞춘 값이라 **낙관적**이고, 테스트 구간 값이 보고할 *성능*입니다. 이 합성데이터에서 두 값의 차이가 얼마나 되는지 위 표로 확인하세요.
- 평균 정밀도(AP)는 **부도율에 따라 값 자체가 달라집니다**(무작위 모형의 값 = 부도율). 부도율이 다른 표본끼리 직접 비교하지 않습니다.

In [ ]:
from creditlab.evaluation import bootstrap_ci, roc_points

ci_auc = bootstrap_ci(auroc, test[TARGET], pd_test, n_boot=500, seed=SEED)
ci_ks = bootstrap_ci(lambda yy, rr: ks_statistic(yy, rr)["ks"], test[TARGET], pd_test, n_boot=500, seed=SEED)
print(f"테스트 AUROC {ci_auc['point']:.3f}  95% 부트스트랩 구간 [{ci_auc['lo']:.3f}, {ci_auc['hi']:.3f}]  (유효 {ci_auc['n_valid']}회, 건너뜀 {ci_auc['skipped']}회)")
print(f"테스트 KS    {ci_ks['point']:.3f}  95% 부트스트랩 구간 [{ci_ks['lo']:.3f}, {ci_ks['hi']:.3f}]")

pts = roc_points(test[TARGET], pd_test)
ks = ks_statistic(test[TARGET], pd_test)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
ax[0].plot(pts["fpr"], pts["tpr"], color="#c8553d"); ax[0].plot([0, 1], [0, 1], "--", color="gray")
ax[0].set_xlabel("FPR (good flagged as bad)"); ax[0].set_ylabel("TPR (bad caught)"); ax[0].set_title(f"ROC  AUROC={a_pd:.3f}")
ax[1].plot(pts["threshold"][1:], pts["tpr"][1:], label="bad cumulative (TPR)", color="#c8553d")
ax[1].plot(pts["threshold"][1:], pts["fpr"][1:], label="good cumulative (FPR)", color="#3d6fc8")
ax[1].axvline(ks["threshold"], ls=":", color="k"); ax[1].set_xlabel("PD threshold"); ax[1].set_title(f"KS={ks['ks']:.3f}"); ax[1].legend()
plt.tight_layout(); plt.show()

**구간이 말하는 것**: 지표는 한 번의 표본에서 나온 *추정치*입니다. 구간이 겹치는 두 모형의 순위를 가리는 일은 위험합니다(단계 14). 부도 건수가 적을수록 구간은 더 넓어집니다.

### Gini 와 AR — 정의를 확인하세요

In [ ]:
y_t = test[TARGET]
print(f"연속 PD     : Gini = {gini(y_t, pd_test):.5f}, CAP 기반 AR = {cap_accuracy_ratio(y_t, pd_test):.5f}")
coarse = -(np.round(sc_test / 10) * 10)             # 점수를 10점 단위로 뭉쳐 동점이 많은 경우
print(f"10점 단위 점수: Gini = {gini(y_t, coarse):.5f}, CAP 기반 AR = {cap_accuracy_ratio(y_t, coarse):.5f}")
print("동점 수:", int(len(coarse) - len(np.unique(coarse))))

동점이 없으면 **AR = Gini = 2·AUROC−1** 입니다. 동점이 많으면 계산 방식(동점을 어떻게 세는가)에 따라 값이 달라질 수 있어, **보고서에 산식을 적어야** 합니다.

## 5. 혼동행렬은 임계값에 따라 달라진다

In [ ]:
pred_acc = None  # ← 부도율 5% 데이터에서 '모두 정상'이라고 찍으면 정확도는? 재현율은?

In [ ]:
rows = [confusion_at(test[TARGET], pd_test, t) for t in [0.05, 0.10, 0.15, 0.25]]
print(pd.DataFrame(rows).round(3).to_string(index=False))

# 부도율이 낮을 때 '모두 정상' 예측의 정확도
base = float(test[TARGET].mean())
allgood = confusion_at(test[TARGET], np.zeros(len(test)), 1.0)
print(f"\n테스트 부도율 {base:.1%}.  '모두 정상' 예측 → 정확도 {allgood['accuracy']:.3f}, 재현율 {allgood['recall']:.3f}")
if pred_acc is not None:
    print("내 예측:", pred_acc)

임계값이 낮을수록 재현율은 오르고 정밀도는 내려갑니다. **정확도 하나로 비교하지 않고, 임계값을 함께 말합니다.** 부도율이 낮은 문제에서 '모두 정상'이 높은 정확도를 얻는다는 점이 정확도를 신용평가의 주 지표로 쓰지 않는 이유입니다.

## 6. 변별력 ≠ 보정

In [ ]:
from creditlab.evaluation import calibration_table, expected_calibration_error, calibration_slope_intercept

ct = calibration_table(test[TARGET], pd_test, n_bins=5)
ct["95% 구간"] = [f"{lo:.1%}~{hi:.1%}" for lo, hi in zip(ct["obs_lo"], ct["obs_hi"])]
print(ct[["bin", "n", "mean_pred", "obs_rate", "95% 구간", "gap"]].round(4).to_string(index=False))
print(f"\nECE = {expected_calibration_error(ct):.4f}  |  보정 기울기·절편 = {calibration_slope_intercept(test[TARGET], pd_test)}")
print(f"Brier = {brier_score(test[TARGET], pd_test):.4f},  log loss = {log_loss_score(test[TARGET], pd_test):.4f}")

# 확률을 2배로 부풀려도 순위는 그대로
p2 = np.clip(2 * pd_test, 0, 1)
ct2 = calibration_table(test[TARGET], p2, n_bins=5)
print(f"\n[PD×2] AUROC {auroc(test[TARGET], p2):.4f} (원래 {a_pd:.4f})  |  ECE {expected_calibration_error(ct2):.4f}  |  평균예측 {p2.mean():.3f} vs 실제 부도율 {base:.3f}")
print("       보정 기울기·절편 =", calibration_slope_intercept(test[TARGET], p2))

- 보정표는 *구간별 평균 예측*과 *실제 부도율*을 나란히 봅니다. 구간의 신뢰구간이 넓다는 것은 **차이가 우연일 수 있다**는 뜻입니다.
- **PD×2** 는 AUROC 가 변하지 않지만 평균 예측이 실제의 두 배가 되고 보정 절편이 크게 음수가 됩니다. 순위 지표는 **보정 오류를 못 봅니다.**
- 보정 기울기가 1 보다 작으면 예측이 지나치게 극단적, 1 보다 크면 너무 보수적입니다. 절편은 기울기와 **함께** 읽습니다.
- 어긋남이 확인되면 재보정(Platt/Isotonic 등)을 검토하며, 이는 Phase 5(단계 12)에서 다룹니다. **테스트 결과를 보고 모형을 고치면 테스트가 더는 테스트가 아닙니다.**

## 7. 기간별 변동

In [ ]:
from creditlab.evaluation import metrics_by_period

q = pd.PeriodIndex(pd.to_datetime(test["application_date"]), freq="Q").astype(str)
bp = metrics_by_period(test, q, TARGET, pd_test)
print(bp[["period", "n", "bads", "bad_rate", "auroc", "gini", "ks"]].round(3).to_string(index=False))

분기별 부도 건수가 적으면 AUROC 가 크게 출렁입니다. **작은 기간의 변동을 곧바로 '성능 저하'로 읽지 않고** 건수와 구간을 함께 보며, 장기적으로 한 방향으로 움직이는지 봅니다.

## 8. ✏️ 실습 — 동점이 있는 AUROC

In [ ]:
# [exercise]
y_ex = np.array([1, 1, 0, 1, 0, 0, 0, 0])
s_ex = np.array([0.8, 0.6, 0.6, 0.5, 0.5, 0.3, 0.2, 0.1])      # 동점(0.6, 0.5)이 있다

auc_mine = None      # ← 부도×정상 쌍을 모두 비교해 구한 AUROC (동점 쌍은 0.5 로 센다)
# 힌트: 부도 점수 목록 · 정상 점수 목록을 나눈 뒤 이중 반복, (b > g) + 0.5*(b == g)

if auc_mine is not None:
    true = auroc(y_ex, s_ex)
    print(f"내 계산 {auc_mine:.4f} | 라이브러리 {true:.4f} :", "맞음" if abs(auc_mine - true) < 1e-9 else "다시 계산")
else:
    print("auc_mine 을 채우고 다시 실행하세요.")

<details>
<summary>정답 예시 (먼저 직접 해 본 뒤 펼치세요)</summary>

```python
bad_s, good_s = s_ex[y_ex == 1], s_ex[y_ex == 0]
auc_mine = float(np.mean([(b > g) + 0.5 * (b == g) for b in bad_s for g in good_s]))
```
</details>

## 9. 퀴즈

In [ ]:
from creditlab.learning import Question, check, grade, show

QUIZ = [
    Question("Q1", "AUROC 정의", "AUROC 0.75 의 의미로 가장 알맞은 것은?",
             {"A": "예측이 75% 맞는다", "B": "무작위로 뽑은 부도 1건과 정상 1건에서 부도의 위험 점수가 더 높을 확률이 75%",
              "C": "부도율이 75% 이다", "D": "상위 75% 를 거절하면 된다"},
             "B", "AUROC 는 부도–정상 쌍의 순위 비교 확률입니다(동점은 0.5).", ("쌍 비교를 떠올려 보세요.",)),
    Question("Q2", "변별력 vs 보정", "모든 예측 PD 에 2를 곱했습니다(1 을 넘으면 1). 어떤 일이 일어날까요?",
             {"A": "AUROC 와 보정이 모두 그대로", "B": "AUROC 는 거의 그대로, 보정은 나빠진다",
              "C": "AUROC 는 반으로 줄고 보정은 그대로", "D": "KS 만 변한다"},
             "B", "순위가 같아 AUROC 는 같지만 평균 예측이 실제의 두 배가 되어 보정이 깨집니다.", ("순위가 바뀌었나요?",)),
    Question("Q3", "정확도의 함정", "부도율 1% 데이터에서 '모두 정상'이라고 예측했습니다. 정확도와 재현율은?",
             {"A": "정확도 99%, 재현율 0%", "B": "정확도 1%, 재현율 100%", "C": "정확도 50%, 재현율 50%", "D": "정확도 99%, 재현율 99%"},
             "A", "정상을 모두 맞히지만 부도는 하나도 못 잡습니다. 그래서 임계값·재현율·정밀도를 함께 봅니다.", ("부도를 몇 건 잡았나요?",)),
    Question("Q4", "개발 vs 테스트", "개발 구간 AUROC 0.80, 테스트 AUROC 0.76 입니다. 보고할 성능은?",
             {"A": "0.80 (더 높으니까)", "B": "0.76 (학습에 쓰지 않은 구간에서의 값, 구간과 함께)", "C": "평균 0.78", "D": "둘 중 마음에 드는 값"},
             "B", "개발 지표는 적합도이고, 테스트 지표가 성능 추정치입니다. 신뢰구간과 함께 보고합니다.", ("어느 구간이 학습에 쓰였나요?",)),
    Question("Q5", "정의 불가", "한 분기 샘플에 부도가 0건입니다. 그 분기의 AUROC 는?",
             {"A": "1.0", "B": "0.5", "C": "0.0", "D": "정의되지 않는다(계산 불가로 표시)"},
             "D", "부도–정상 쌍이 없으므로 AUROC 는 정의되지 않습니다. 숫자를 만들어 내지 않습니다.", ("쌍이 몇 개 만들어지나요?",)),
]

answers = {"Q1": None, "Q2": None, "Q3": None, "Q4": None, "Q5": None}  # ← "A"~"D"

for q in QUIZ:
    show(q)
    print()

In [ ]:
for q in QUIZ:
    check(q, answers[q.qid])
if all(v is not None for v in answers.values()):
    r = grade(QUIZ, answers)
    print("\n점수:", r["score"], "/", r["total"], "| 통과(75% 기준):", r["passed"], "| 다시 볼 개념:", r["concepts_to_review"])

## 10. 과제와 정리
**과제**
1. 테스트 구간을 **분기 3개**로 나누어 AUROC·KS 와 부트스트랩 구간을 표로 만들고, 구간이 겹치는지 보세요. 한 분기만 낮다면 *우연일 가능성*을 어떻게 판단할지 두세 문장으로 쓰세요.
2. 임계값을 0.05~0.30 으로 움직이며 정밀도·재현율 곡선을 그리고, **승인율 80%** 가 되는 컷오프에서의 혼동행렬을 구해 보세요. (컷오프 선택은 정책입니다.)
3. 보정 구간 수를 5·10·20 으로 바꿔 ECE 가 얼마나 변하는지 확인하고, 왜 ECE 만으로 판단하면 위험한지 쓰세요.

**체크포인트**: 퀴즈 4/5 이상 · 동점 AUROC 실습 통과 · 점수와 PD 의 AUROC 가 같음을 확인

**다음 레슨**: 레슨 4-2 — PSI 로 입력·점수 분포가 변했는지 감시하고, PSI 가 못 보는 변화(개념 변화)를 구분합니다.

**참고자료**
- Hand, D. J., "Measuring classifier performance: a coherent alternative to the area under the ROC curve", *Machine Learning* 77 (2009).
- Saito, T. & Rehmsmeier, M., "The precision-recall plot is more informative than the ROC plot…", *PLOS ONE* (2015).
- Thomas, L., *Consumer Credit Models* (Oxford, 2009).
- 이론: `docs/theory/stage_09.html`